### Questoin 2, Part 1c

In [1]:
# import the necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sktime.forecasting.compose import ColumnEnsembleForecaster
from sktime.forecasting.arima import AutoARIMA
from sktime.forecasting.base import ForecastingHorizon
from sktime.split import temporal_train_test_split
from sktime.performance_metrics.forecasting import mean_absolute_percentage_error

In [2]:
# read the cleaned version of the dataset
df = pd.read_csv('AirtrafficA4_cleaned.csv', index_col=0)
df.index = pd.to_datetime(df.index).to_period('M')
df

,TOTAL DEPARTURES,HOURS FLOWN,KILOMETRE FLOWN,PASSENGERS CARRIED,AVAILABLE SEAT KILOMETRE (IN THOUSAND),FREIGHT CARRIED (IN TONNE),MAIL CARRIED (IN TONNE)
2013-01,10552.0,18655.0,10112.0,1408012.0,1820105.0,6465.0,0.0
2013-02,9873.0,17374.0,9439.0,1341210.0,1698930.0,6235.0,0.0
2013-03,11393.0,20093.0,11028.0,1423569.0,1984886.0,6505.0,0.0
2013-04,11426.0,20084.0,11090.0,1511094.0,1996084.0,5903.0,0.0
2013-05,11885.0,20779.0,11533.0,1685168.0,2075882.0,7345.0,0.0
...,...,...,...,...,...,...,...
2023-04,48752.0,84232.0,42615.0,7406440.0,8005648.0,19432.8,2102.9
2023-05,50956.0,87917.0,44505.0,8109626.0,8375201.0,24165.1,2102.4
2023-06,49989.0,86217.0,43739.0,7893296.0,8254272.0,23522.6,2383.0
2023-07,52127.0,90528.0,45404.0,7674890.0,8577184.0,24885.8,2585.0


In [3]:
# Split your data into train and test sets
df_train, df_test = temporal_train_test_split(df, test_size=.20)

In [4]:
# Define a forecaster for each column using AutoARIMA
forecaster = ColumnEnsembleForecaster([
    (
        col.lower().replace(' ', '_').replace('(', '').replace(')', ''), 
        AutoARIMA(suppress_warnings=True), 
        col,
    ) for col in df.columns
])

In [5]:
# Fit the global model on the training data
forecaster.fit(df_train)

ColumnEnsembleForecaster(forecasters=[('total_departures',
                                       AutoARIMA(suppress_warnings=True),
                                       'TOTAL DEPARTURES'),
                                      ('hours_flown',
                                       AutoARIMA(suppress_warnings=True),
                                       'HOURS FLOWN'),
                                      ('kilometre_flown',
                                       AutoARIMA(suppress_warnings=True),
                                       'KILOMETRE FLOWN'),
                                      ('passengers_carried',
                                       AutoARIMA(suppress_warnings=True),
                                       'PASSENGERS CARRIED'),
                                      ('available_seat_kilometre_in_thousand',
                                       AutoARIMA(suppress_warnings=True),
                                       'AVAILABLE SEAT KILOMETRE (IN '
                                       'THOUSAND)'),
                                      ('_freight_carried_in_tonne',
                                       AutoARIMA(suppress_warnings=True),
                                       ' FREIGHT CARRIED (IN TONNE)'),
                                      ('mail_carried_in_tonne',
                                       AutoARIMA(suppress_warnings=True),
                                       'MAIL CARRIED (IN TONNE)')])

In [6]:
# Define the forecasting horizon (predicting into the test set)
fh = ForecastingHorizon(df_test.index, is_relative=False)

In [7]:
# Predict for each series
df_pred = forecaster.predict(fh)

In [8]:
# report the accuracy of the model
# it is represented as the Mean Absolute Percentage Error
mean_absolute_percentage_error(df_test, df_pred)

0.3997055093067567

In [9]:
# We report a MAPE of approximately 0.4%, which is pretty good

In [10]:
# Now, we can fit the forecaster on the entire dataset
forecaster.fit(df)

ColumnEnsembleForecaster(forecasters=[('total_departures',
                                       AutoARIMA(suppress_warnings=True),
                                       'TOTAL DEPARTURES'),
                                      ('hours_flown',
                                       AutoARIMA(suppress_warnings=True),
                                       'HOURS FLOWN'),
                                      ('kilometre_flown',
                                       AutoARIMA(suppress_warnings=True),
                                       'KILOMETRE FLOWN'),
                                      ('passengers_carried',
                                       AutoARIMA(suppress_warnings=True),
                                       'PASSENGERS CARRIED'),
                                      ('available_seat_kilometre_in_thousand',
                                       AutoARIMA(suppress_warnings=True),
                                       'AVAILABLE SEAT KILOMETRE (IN '
                                       'THOUSAND)'),
                                      ('_freight_carried_in_tonne',
                                       AutoARIMA(suppress_warnings=True),
                                       ' FREIGHT CARRIED (IN TONNE)'),
                                      ('mail_carried_in_tonne',
                                       AutoARIMA(suppress_warnings=True),
                                       'MAIL CARRIED (IN TONNE)')])

In [11]:
# make forecasts for the next 12 months
fh_forward = ForecastingHorizon(np.arange(1, 13), is_relative=True)

In [12]:
df_forecast = forecaster.predict(fh=fh_forward)
df_forecast

,TOTAL DEPARTURES,HOURS FLOWN,KILOMETRE FLOWN,PASSENGERS CARRIED,AVAILABLE SEAT KILOMETRE (IN THOUSAND),FREIGHT CARRIED (IN TONNE),MAIL CARRIED (IN TONNE)
2023-09,52296.374594,90165.313845,45566.494584,7.929540e+06,8.657825e+06,23228.890184,2608.1
2023-10,51636.912942,88886.134633,44966.506325,7.929540e+06,8.538089e+06,22541.146961,2608.1
2023-11,51209.086207,87963.600083,44575.702467,7.929540e+06,8.463030e+06,21942.396766,2608.1
2023-12,50931.532972,87404.872754,44321.151395,7.929540e+06,8.415976e+06,21464.094403,2608.1
2024-01,50751.469895,87102.005932,44155.348915,7.929540e+06,8.386479e+06,21120.308287,2608.1
2024-02,50634.653704,86951.928510,44047.353054,7.929540e+06,8.367988e+06,20911.124810,2608.1
2024-03,50558.869025,86883.811484,43977.009678,7.929540e+06,8.356396e+06,20826.250085,2608.1
2024-04,50509.703601,86855.925759,43931.191345,7.929540e+06,8.349129e+06,20848.480955,2608.1
2024-05,50477.807459,86846.114941,43901.347458,7.929540e+06,8.344574e+06,20956.811917,2608.1
2024-06,50457.114788,86843.605533,43881.908567,7.929540e+06,8.341718e+06,21129.032830,2608.1


In [13]:
# save the forecasted data
df_forecast.to_csv('AirtrafficA4_forecast.csv')